In [1]:
import pandas as pd
import torch
from  torch.optim import AdamW, Adam, SGD
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments, get_linear_schedule_with_warmup
from datasets import Dataset
from sklearn.metrics import accuracy_score
import gc
from math import ceil
from transformers import EarlyStoppingCallback
from utils.config import QUESTION_TYPES

In [2]:
new_dataset_path= 'Model_dataset/real_world_data.csv'
old_dataset_path= 'Model_dataset/synthetic_question_ans_data-v2.csv'

# q type classifer,
q_type_model_path= 'model/fine_tuned_question_classifier_model_lite-default'
q_type_model_result= '.temp/model_results/q_types_model_lite_results'
q_type_model= '.temp/model/fine_tuned_question_classifier_model_lite'



# Question type classsifier

### Preprocessing

In [ ]:
data_ratio= {
    "old":20,
    "new":80
    }

new_data_df= pd.read_csv(new_dataset_path)
new_data_df= new_data_df[["question", "question_type"]]
new_data_df.info()

old_data_df= pd.read_csv(old_dataset_path)
old_data_df= old_data_df[["question", "question_type"]]
old_data_df.info()

In [ ]:
columns_in_new_df= new_data_df["question_type"].unique()
print(f"columns_in_new_df :{columns_in_new_df}")

columns_in_old_df= old_data_df["question_type"].unique()
print(f"columns_in_old_df :{columns_in_old_df}")

In [ ]:
new_data_df.drop_duplicates(inplace= True)
new_data_df.info()

old_data_df.drop_duplicates(inplace= True)
old_data_df.info()

#### combaing new and old data

In [ ]:
new_data_len= len(new_data_df)
total_len= ceil(new_data_len/(data_ratio["new"]/100))
old_data_len= ceil(total_len- new_data_len)
print(total_len)
old_data_len

In [ ]:
temp_df= pd.DataFrame()
while True:
    temp_df= old_data_df.sample(old_data_len)
    columns_in_old_df= temp_df["question_type"].unique()

    if set(columns_in_old_df)== set(columns_in_new_df):
        break
temp_df.info()

In [ ]:
df= pd.concat([new_data_df, temp_df], ignore_index=True)
df.drop_duplicates(inplace= True)
df.info()

#### adding labels

In [ ]:
# adding labels
label_mapping = {key: index for index, key in enumerate(QUESTION_TYPES)}
df['label'] = df['question_type'].map(label_mapping)

# droping unused column
df.drop('question_type', axis=1,  inplace= True)
df.head()

In [ ]:
df= df.sample(frac=1).reset_index(drop=True)
df.info()

In [ ]:
df.head(5)

### Retraing Preparations:

In [12]:
#convert to hugging face dataset
dataset= Dataset.from_pandas(df)

#Split the data into train and test sets (80-20 split)
dataset_split = dataset.train_test_split(test_size=0.2)

# Access train and test splits
train_dataset = dataset_split['train']
test_dataset = dataset_split['test']

In [13]:
tokenizer = DistilBertTokenizerFast.from_pretrained(q_type_model_path)

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['question'], padding= "max_length", truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set the format to PyTorch tensors
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

In [15]:
# Mapping lebel and id
id2label = {v: k for k, v in label_mapping.items()}  # Map IDs to label names
label2id = {k: v for k, v in label_mapping.items()}  # Map label names to IDs

In [16]:
total_training_steps = (len(train_dataset) // (16 * 2)) * 4  # Example calculation: adjust as needed
warmup_steps = int(0.1 * total_training_steps)

# Retraning

# using default optimizer

In [ ]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_path,
        num_labels= len(columns_in_old_df),
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

In [18]:
training_args = TrainingArguments(
    output_dir=q_type_model_result + "-default",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs= 50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

- seems like epoch 3 will be best for prediction

### Model evaluation and Saving

In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(q_type_model+"-default")
tokenizer.save_pretrained(q_type_model+"-default")

In [22]:
del trainer, model

# Using AdamW optimiser with linear scheduler with warmup

In [ ]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_path,
        num_labels= len(columns_in_old_df),
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

In [24]:
training_args = TrainingArguments(
    output_dir=q_type_model_result + "-AdamW",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs= 50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
)

In [25]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = AdamW(model.parameters(), lr=5e-5, eps=1e-8)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [ ]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

- seems like epoch 4 will be best for prediction

In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(q_type_model+"-AdamW")
tokenizer.save_pretrained(q_type_model+"-AdamW")

In [29]:
del trainer, model


# Using Adam

In [ ]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_path,
        num_labels= len(columns_in_old_df),
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

In [31]:
training_args = TrainingArguments(
    output_dir=q_type_model_result + "-Adam",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs= 50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
)

In [32]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = Adam(model.parameters(), lr=3e-5)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [ ]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

- seems like epoch 6 will be best for prediction

In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(q_type_model+"-Adam")
tokenizer.save_pretrained(q_type_model+"-Adam")

In [36]:
del trainer, model


# Using SGD

In [ ]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_path,
        num_labels= len(columns_in_old_df),
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

In [38]:
training_args = TrainingArguments(
    output_dir=q_type_model_result + "-SGD",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs= 50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
)

In [39]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [ ]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

- seems like epoch 6 will be best for prediction

In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(q_type_model+"-SGD")
tokenizer.save_pretrained(q_type_model+"-SGD")

In [ ]:
del trainer, model
